# Replication on Gemma 3 27B: flow map, Stage 2, Stage 2b (preregistered: `docs/preregistration_replication_27b.md`)

The three 12B tests, unchanged except for layers scaled to the deeper model: where the first step goes on the way to the answer (flow map), which components write the answer (Stage 2), and whether removing them while keeping the names leaves a correct read-out (Stage 2b).

**Before you press Run all:**
1. Upload the new `geomap_colab.zip` to Google Drive at **`MyDrive/geomap/geomap_colab.zip`** (replace the old one).
2. Colab secrets (key icon), notebook access on: **`HF_TOKEN`** (required), `SLACK_WEBHOOK_URL` (optional).
3. Runtime → Change runtime type → **H100** or **G4 (RTX PRO 6000)**, or an **A100 80 GB**. The whole 27B model (54 GB) must fit on the GPU: a 40 GB A100 will not work for these tests. Then **Run all**. About 2–4 hours.

Each test saves to Drive as it goes and is skipped once complete, so after a disconnect just **Run all** again. At the end `replication_27b_results.zip` downloads (also saved in `MyDrive/geomap/`). Cell 8 can be run on its own at any time to zip what is finished so far.

In [ ]:
# 1. GPU check: the whole model must fit (no CPU offload for these tests)
import subprocess, torch, os
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> H100 / G4"
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
assert gb > 70, f"Only {gb:.0f} GB of GPU memory: pick H100, G4 or A100 80 GB (the 27B model needs ~60 GB)"

In [ ]:
# 2. Code from Drive; results live ON Drive (saved as they are produced, kept across disconnects)
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, zipfile
D = "/content/drive/MyDrive/geomap"
BUNDLE, RES = f"{D}/geomap_colab.zip", f"{D}/results"
assert os.path.exists(BUNDLE), f"Upload geomap_colab.zip to {BUNDLE}"
shutil.rmtree("/content/geomap", ignore_errors=True)
zipfile.ZipFile(BUNDLE).extractall("/content/geomap")
if not os.path.exists(RES):
    shutil.copytree("/content/geomap/results", RES)
assert "def layer_grid" in open("/content/geomap/src/geomap/flow.py").read(), "Old bundle: upload the new geomap_colab.zip"
shutil.rmtree("/content/geomap/results")
os.symlink(RES, "/content/geomap/results")
os.chdir("/content/geomap")
for sub in ("flow", "components"):
    p = f"{RES}/{sub}"
    print(sub, "already saved:", sorted(f for f in os.listdir(p) if "27b" in f) if os.path.exists(p) else "nothing yet")

In [ ]:
# 3. Dependencies, PyTorch only (Colab's TensorFlow breaks transformers' Gemma 3 import)
!pip -q install "transformers==4.57.3" nnsight accelerate
import sys
os.environ.update(PYTHONPATH="/content/geomap/src", USE_TF="0", USE_FLAX="0", USE_TORCH="1",
                  TRANSFORMERS_NO_TF="1", TF_CPP_MIN_LOG_LEVEL="3")
sys.path.insert(0, "/content/geomap/src")
!python -c "from transformers.models.gemma3.modeling_gemma3 import Gemma3ForConditionalGeneration; print('Gemma 3 import OK')"

In [ ]:
# 4. Secrets: Hugging Face (required) and Slack (optional)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami
print("Hugging Face:", whoami(token=os.environ["HF_TOKEN"])["name"])
try:
    os.environ["SLACK_WEBHOOK_URL"] = userdata.get("SLACK_WEBHOOK_URL")
except Exception:
    pass
import sys
sys.modules.pop("geomap", None)  # a stale or wrong "geomap" import must not shadow the code
sys.path.insert(0, "/content/geomap/src")
from geomap.checkpoint import notify
notify("[Replication 27B] Colab run started (or resumed)")

In [ ]:
# 5. Flow map (the ~5 GB residual cache stays on the Colab disk; the results go to Drive)
!cd /content/geomap && PYTHONPATH=/content/geomap/src USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 python scripts/flow_test.py --model gemma3-27b --batch-size 8 --acts-dir /content/flow_acts 2>&1 | grep -v -i warning

In [ ]:
# 6. Stage 2: attribution on the discovery set (picks the components for Stage 2b), then its mean-ablation test
!cd /content/geomap && PYTHONPATH=/content/geomap/src USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 python scripts/components_test.py --model gemma3-27b --batch-size 8 2>&1 | grep -v -i warning

In [ ]:
# 7. Stage 2b: twin-mean removal of Stage 2's top 20, swap, attribution under removal
!cd /content/geomap && PYTHONPATH=/content/geomap/src USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 python scripts/components2_test.py --model gemma3-27b --batch-size 8 2>&1 | grep -v -i warning

In [ ]:
# 8. One zip of the results + the results log (not the caches), on Drive and downloaded here
import glob
with zipfile.ZipFile("/content/replication_27b_results.zip", "w") as z:
    for sub, pat in (("flow", "gemma3-27b_flow*"), ("components", "gemma3-27b_components*")):
        for f in glob.glob(f"{RES}/{sub}/{pat}"):
            if not f.endswith((".npz", "_acts.pt")):
                z.write(f, f"results/{sub}/" + os.path.basename(f))
    z.write(f"{RES}/RESULTS.md", "results/RESULTS.md")
    print("\n".join(z.namelist()))
shutil.copy("/content/replication_27b_results.zip", f"{D}/replication_27b_results.zip")
notify("[Replication 27B] zip saved: MyDrive/geomap/replication_27b_results.zip")
from google.colab import files
files.download("/content/replication_27b_results.zip")